# MINRES: in between GMRES and CG

Elisa Klunder (s5190940) and Dries Wedda (s4745329), 19.01.2026

# Introduction

# MINRES description

# Implementation

In [1]:
import numpy as np
from scipy.linalg import solve_triangular


In [2]:
def make_symmetric_matrix(size: int) -> np.ndarray:
    """Create a symmetric matrix `A` with elements sampled from N(0,1)."""
    A = np.empty((size, size))
    for i in range(size):
        for j in range(i, size):           # all elements j >= i
            A[j, i] = A[i, j] = np.random.normal()
    return A


def make_target(size: int) -> np.ndarray:
    """Create a target vector `b` with elements sampled from N(0,1)."""
    return np.random.normal(size=size)

In [28]:
def lanczos(A: np.ndarray, b: np.ndarray, k: int) -> list:
    # TODO reduce code duplication
    #       improve efficiency so no recomputations, let it run 1 iteration in function
    #       and store results in minres()
    # TODO list or np array for alpha and beta?
    q = [b / np.linalg.norm(b)]
    
    v = A @ q[0]
    alpha = [np.dot(v, q[0])]     # alpha_1 = v.T q_1
    v -= alpha[0] * q[0]
    beta = [np.linalg.norm(v)]    # beta_1 = ||v||
    q.append(v / beta[0])         # q_2 = v / beta_1

    for j in range(1, k):
        v = A @ q[j]
        alpha.append(np.dot(v, q[j]))
        v = v - alpha[j] * q[j] - beta[j - 1] * q[j - 1]
        beta.append(np.linalg.norm(v))
        q.append(v / beta[j])    # q_j = v / beta_j
    
    return np.array(q).T, alpha, beta


def make_hessenberg(alpha: list, beta: list, k: int) -> np.ndarray:
    H = np.zeros((k + 1, k))
    H[0, 0] = alpha[0]
    H[1, 0] = beta[0]
    for idx in range(1, k):
        H[idx, idx] = alpha[idx]
        H[idx - 1, idx] = beta[idx - 1]
        H[idx + 1, idx] = beta[idx]
    return H

def minres(A: np.ndarray, b: np.ndarray):
    max_iter = b.shape[0]
    for k in range(1, max_iter + 1):
        print(f"k = {k}")

        Q, alpha, beta = lanczos(A, b, k)
        H = make_hessenberg(alpha, beta, k)
        V, R = np.linalg.qr(H)                  # TODO add Givens rotations

        z = np.zeros(k + 1)                # ||b||*e_1 for R^{k+1}
        z[0] = np.linalg.norm(b)

        y = solve_triangular(R, V.T @ z, lower=False)    # y = R^-1 V.T z --> Ry = V.T z
                                            # note R is upper triangular!
        x = Q[:,:k] @ y
        
        print("residual:", np.linalg.norm(A @ x - b))


np.random.seed(0)
m = 10
A = make_symmetric_matrix(m)
b = make_target(m)

minres(A, b)


k = 1
residual: 2.251071475860654
k = 2
residual: 1.7291612887014605
k = 3
residual: 1.6652536888761518
k = 4
residual: 1.3853268317964114
k = 5
residual: 0.5542088443066202
k = 6
residual: 0.46118568356149675
k = 7
residual: 0.4570308020635816
k = 8
residual: 0.24395829823900378
k = 9
residual: 0.051213900289790934
k = 10
residual: 1.957625652324875e-12


# Comparison

# Extension